# 3.4. Porównanie: KNN, DBSCAN i k-means na tych samych danych

Ten notebook porządkuje trzy metody z modułu 3.

| Metoda | Co robi? | Główna idea |
|---|---|---|
| KNN | klasyfikuje punkt testowy | głosowanie sąsiadów |
| DBSCAN | znajduje klastry i szum | gęste komponenty |
| k-means | znajduje centroidy | minimalizacja SSE |

Wszystkie używają odległości, ale odpowiadają na inne pytania.


## W KNN notebooku użyjemy

✓ Macierzy odległości D

✓ Macierzy kNN

## W DBSCAN notebooku użyjemy

✓ Macierzy odległości D

✓ Macierzy ε-sąsiedztwa

## W KMeans notebooku użyjemy

✓ Punktów danych X

✓ Centroidów

✗ Macierzy sąsiedztwa
✗ Grafów
✗ kNN

## Do jakiej rodziny należy metoda?

PCA:
→ metody projekcyjne

t-SNE:
→ metody podobieństw

UMAP:
→ metody grafowe

KNN:
→ metody sąsiedztwa

DBSCAN:
→ metody gęstościowe

KMeans:
→ metody centroidowe

LDA:
→ metody dyskryminacyjne

SVM:
→ metody maksymalnego marginesu

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt
from IPython.display import display, Markdown

np.set_printoptions(precision=3, suppress=True)

def pairwise_distances(X):
    X = np.asarray(X, dtype=float)
    diff = X[:, None, :] - X[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=2))

def show_matrix(M, labels, title, fmt=".2f", cmap="viridis", vmin=None, vmax=None):
    df = pd.DataFrame(M, index=labels, columns=labels)
    display(df.style.format(fmt))
    fig, ax = plt.subplots(figsize=(4.8, 4.1))
    im = ax.imshow(M, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_title(title)
    for i in range(len(labels)):
        for j in range(len(labels)):
            txt = f"{M[i,j]:.1f}" if abs(M[i,j]) < 10 else f"{M[i,j]:.0f}"
            ax.text(j, i, txt, ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()


def plot_points(X, labels, y=None, title="punkty", annotate=True, extra_points=None):
    X = np.asarray(X, dtype=float)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    if y is None:
        ax.scatter(X[:,0], X[:,1], s=90)
    else:
        y = np.asarray(y)
        for val in np.unique(y):
            mask = y == val
            ax.scatter(X[mask,0], X[mask,1], s=90, label=str(val))
        ax.legend(title="etykieta")
    if annotate:
        for i, lab in enumerate(labels):
            ax.text(X[i,0]+0.05, X[i,1]+0.05, lab, fontsize=12, weight='bold')
    if extra_points:
        for name, p in extra_points.items():
            ax.scatter([p[0]], [p[1]], s=130, marker='x')
            ax.text(p[0]+0.05, p[1]+0.05, name, fontsize=12, weight='bold')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    plt.show()


def adjacency_eps(D, eps):
    A = ((D <= eps) & (D > 0)).astype(int)
    return A


def knn_directed(D, k):
    n = D.shape[0]
    A = np.zeros((n,n), dtype=int)
    for i in range(n):
        order = np.argsort(D[i])
        neigh = [j for j in order if j != i][:k]
        A[i, neigh] = 1
    return A


def sym_or(A):
    return ((A + A.T) > 0).astype(int)


def sym_mutual(A):
    return ((A + A.T) == 2).astype(int)


def plot_graph(X, labels, A, title="graf", directed=False):
    X = np.asarray(X, dtype=float)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.scatter(X[:,0], X[:,1], s=100)
    for i, lab in enumerate(labels):
        ax.text(X[i,0]+0.05, X[i,1]+0.05, lab, fontsize=12, weight='bold')
    n = len(labels)
    for i in range(n):
        for j in range(n):
            if A[i,j] and (directed or i < j):
                if directed:
                    ax.annotate("", xy=X[j], xytext=X[i], arrowprops=dict(arrowstyle="->", lw=1.4, alpha=0.65))
                else:
                    ax.plot([X[i,0], X[j,0]], [X[i,1], X[j,1]], lw=1.5, alpha=0.65)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    plt.show()

from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_blobs


## 1. Wspólny zbiór danych


In [ ]:
X, y_true = make_blobs(n_samples=120, centers=[[-2,0],[2,0],[0,2.6]], cluster_std=[0.45,0.55,0.5], random_state=8)
# dodajemy kilka punktów szumu
noise = np.array([[0,-2.2], [3.5,2.5], [-3.2,2.2], [0.2, -1.7]])
X_all = np.vstack([X, noise])
y_train = np.r_[y_true, [-1]*len(noise)]
fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(X_all[:,0], X_all[:,1], s=35)
ax.grid(True, alpha=0.3)
ax.set_title("Wspólny zbiór danych")
plt.show()


## 2. k-means: centroidy

Zakładamy liczbę klastrów:

$$
K=3.
$$

Metoda znajdzie trzy centroidy, nawet jeśli istnieją punkty szumu.


In [ ]:
km = KMeans(n_clusters=3, random_state=0, n_init=10).fit(X_all)
fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(X_all[:,0], X_all[:,1], c=km.labels_, s=35)
ax.scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1], s=220, marker='X', edgecolor='k')
ax.grid(True, alpha=0.3)
ax.set_title("k-means: zawsze przypisuje każdy punkt do centroidu")
plt.show()


## 3. DBSCAN: gęstość i szum

DBSCAN może oznaczyć punkty jako noise:

$$
\text{label}=-1.
$$


In [ ]:
db = DBSCAN(eps=0.75, min_samples=5).fit(X_all)
fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(X_all[:,0], X_all[:,1], c=db.labels_, s=35)
ax.grid(True, alpha=0.3)
ax.set_title("DBSCAN: klastry gęstościowe i noise")
plt.show()
print("etykiety DBSCAN:", sorted(set(db.labels_)))


## 4. KNN: klasyfikacja punktu testowego

KNN wymaga etykiet klas w danych treningowych. Tutaj użyjemy tylko punktów z klasami 0, 1, 2 i sklasyfikujemy nowy punkt.


In [ ]:
mask_train = y_train != -1
X_train = X_all[mask_train]
y_train_clean = y_train[mask_train]
query = np.array([[0.1, 1.7]])
fig, axes = plt.subplots(1,3, figsize=(14,4.2), sharex=True, sharey=True)
xx, yy = np.meshgrid(np.linspace(-3.8, 4.0, 160), np.linspace(-2.7, 3.6, 150))
grid = np.c_[xx.ravel(), yy.ravel()]
for ax, k in zip(axes, [1,5,15]):
    clf = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train_clean)
    Z = clf.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.22)
    ax.scatter(X_train[:,0], X_train[:,1], c=y_train_clean, s=28, edgecolor='k', linewidth=0.2)
    pred = clf.predict(query)[0]
    ax.scatter(query[:,0], query[:,1], s=180, marker='x')
    ax.set_title(f"KNN k={k}, predykcja X*={pred}")
    ax.grid(True, alpha=0.25)
plt.show()


## 4b. Interaktywny panel KNN: granica decyzyjna dla różnych $k$

KNN jest bardzo czuły na liczbę sąsiadów.

- Małe $k$ daje bardzo lokalną, poszarpaną granicę.
- Duże $k$ wygładza decyzję, ale może zgubić lokalne szczegóły.

To jest ten sam motyw co w `0_1`: parametr $k$ definiuje lokalność.


In [ ]:
import plotly.graph_objects as go
from pathlib import Path

html_dir = Path('../html')
html_dir.mkdir(parents=True, exist_ok=True)

k_values = [1, 3, 5, 9, 15, 31]
active_idx = k_values.index(5)
fig = go.Figure()

for idx, k_value in enumerate(k_values):
    clf = KNeighborsClassifier(n_neighbors=k_value).fit(X_train, y_train_clean)
    Z = clf.predict(grid).reshape(xx.shape)
    visible = idx == active_idx
    pred = clf.predict(query)[0]
    fig.add_trace(go.Contour(
        x=xx[0], y=yy[:, 0], z=Z,
        contours_coloring='heatmap', opacity=0.25, showscale=False,
        visible=visible, name=f'obszary KNN k={k_value}',
        hoverinfo='skip'
    ))

fig.add_trace(go.Scatter(
    x=X_train[:, 0], y=X_train[:, 1], mode='markers',
    marker=dict(size=7, color=y_train_clean), name='dane treningowe'
))
fig.add_trace(go.Scatter(
    x=query[:, 0], y=query[:, 1], mode='markers+text', text=['X*'],
    textposition='top center', marker=dict(size=16, symbol='x'), name='punkt testowy'
))

steps = []
for idx, k_value in enumerate(k_values):
    clf = KNeighborsClassifier(n_neighbors=k_value).fit(X_train, y_train_clean)
    pred = clf.predict(query)[0]
    visible = [False] * len(k_values) + [True, True]
    visible[idx] = True
    steps.append(dict(
        method='update',
        label=str(k_value),
        args=[{'visible': visible}, {'title': f'KNN: k={k_value}, predykcja punktu X*={pred}'}]
    ))

fig.update_layout(
    title='KNN: k=5, predykcja punktu testowego',
    sliders=[dict(active=active_idx, currentvalue={'prefix': 'k = '}, steps=steps)],
    xaxis_title='x1', yaxis_title='x2', height=560
)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.write_html(html_dir / '3_4_knn_k_slider_granica_decyzyjna.html', include_plotlyjs='cdn')
fig.show()


## 6. Podsumowanie

- **k-means** jest dobry, gdy chcemy centroidy i zakładamy kuliste klastry.
- **DBSCAN** jest dobry, gdy interesuje nas gęstość, nieregularne kształty i szum.
- **KNN** jest dobry, gdy mamy etykiety i chcemy sklasyfikować nowy punkt przez lokalne sąsiedztwo.
